# 固定 S/Z 监督实验：实现阶段，实际预检/拟合尚未授权

单一机制：同一个原共享8维表示之后追加同9参数零回归头，B监督总量S，C监督日夜contrast Z。原39/16输入、973参数子集、3014/20741、原H1/H5目标与账户政策保留。

[冻结计划](plan.json) · [设计独评](design_draft_review.json) · [实现合同](implementation_contract.json) · [工作流配置](workflow.yaml)


## 原生 Qlib 入口与分阶段对象边界

获准的预检会执行 `qlib.init` → 在无全局写入 Recorder 时 `R.get_recorder`/`Recorder.load_object` 只读已核验白名单 → `DatasetH.prepare(col_set=DataHandler.CS_RAW)` → 严格日期/原始8字段证明后 `DataHandlerLP.from_df`/`DatasetH`（85→87列）→ `config(dump_all=True,recursive=True)` → `with R.start` 保存与直接 prepare 读回。禁止 `setup_data` 回载和先加载全时期对象再裁剪。

原PG instrument是SOXL_SOXX_CASH；C来源是SOXL_SOXX。严格验证完整3014日期顺序与原目标后才按位置赋值，不做inner join。C Recorder artifact `labels/fixed_H5_components_S_Z` 的实际列组是 `original`(8) 和 `fixed_day_night_label`(11)，只取后者S/Z。

正式运行先封存六模型SHA，才读取原946行评价Dataset和944个账户区间；预测不要求监督列。关闭全局R.start后，以同一明确Recorder低层start_run/end_run包裹评价与原生回放，避免source reader/qlib.init冲突。


In [ ]:
import os, json
from pathlib import Path
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
CARD = ROOT / "research/stage1_day_night_supervision_20261018"
plan = json.loads((CARD / "plan.json").read_text())
api_chain = ["qlib.init", "R.get_recorder / Recorder.load_object(exact verified whitelist)", "DatasetH.prepare(DataHandler.CS_RAW)", "DataHandlerLP.from_df -> DatasetH.config(dump_all=True, recursive=True)", "R.start / Recorder.save_objects / direct prepare roundtrip"]
prefit = os.environ.get("DAY_NIGHT_SUPERVISION_PREFLIGHT") == "1"
execute = os.environ.get("DAY_NIGHT_SUPERVISION_EXECUTE") == "1"
if prefit and execute:
    raise RuntimeError("Preflight and fit stages must be separately approved")
if prefit or execute:
    from src.workflow.day_night_supervision import prepare, run
    display({"approved_workflow_API_chain": api_chain, "phase": "preflight" if prefit else "fit_then_accounts"})
    result = (prepare if prefit else run)(root=ROOT, card=CARD)
    display({k: result.get(k) for k in ["decision", "recorder_id", "new_fits", "new_native_accounts", "gates", "nas_evidence"]})
else:
    display({"status": "implementation_only", "future_authorized_API_chain": api_chain, "real_source_objects_loaded": 0, "fits": 0, "accounts": 0})


## 配对、费用与证据含义

两套目标只在获准训练域预检中各拟合一次float64均值/sample std(ddof=1)，标准化后仅一次float32转换；六fit复用同一冻结常数，不做clip/epsilon救援。原combined loss后才加0.02×half-MSE；仍一次combined backward、全982参数global clip2、AdamW。零head首步直接encoder梯度为0，但head梯度可经globalclip改变private更新，不能将结果归于单纯encoder语义。

原目标还是H1 open→next-open allocation、H5 tail/alpha/cash与原共同成熟筛选。S/Z来自已独评接受的July24分配调整研究价格平面，不是现金股利账本、原daily-reset alpha或实际09:47成交标签。

账户预算为2个等权ensemble×3个费用=6条独立自身状态政策路径/6新native，另复用15本参考，共21比较与15预先固定CI。所有成本/seed保留，MSE/AUC不是账户执行门。0新市场来源查询不等于完全不读账户价格：六模型屏障后获准解码固定raw_points/market，并生成/读取本卡Qlib execution provider；不构造S/Z评价标签。


In [ ]:
display(Markdown((CARD / "conclusion.md").read_text()))


# 固定 S/Z 监督：实现与唯一合成合同完成

新workflow保留原PG3014×85主训练和20741×20辅助，只在严格证明C来源instrument/完整日期顺序/原8字段与digest后追加独立S/Z监督组到87列。原39/16输入和全部原目标不变；来源artifact名与实际fixed_day_night_label列组明确分开，原闲置feature_masks字典保留但extra_features仍为空。

模型保持原973参数子集，在同一次primary共享8状态后加9参数零head；B监督S、C监督Z。原combined loss之后才加固定0.02×half-MSE，原aux采样、一次global clip2及AdamW保留；新head参与全局范数，不能声称private更新受保护。推理不读监督目标，训练图不封存。

唯一集中synthetic最终句柄55347 exit0，1 passed / 2.13s。保留首19539仅fixture频率metadata失败、52338前版通过，以及原闲置mask兼容修正后的55347最终通过履历。实际19列来源schema、PG85→87、原index/参数/RNG、尺度、推理隔离及globalclip耦合都纳入同一合同；没有调用真实model.fit或optimizer.step。

当前真实source对象/标签/模型读取=0、真实梯度=0、无拟合预检=0、fit/account=0。六模型SHA封存后才允许原946行评价Dataset与6成本独立路径/native、15参考及15CI；0新市场来源查询不等于否认已授权账户阶段对固定raw价及派生Qlib execution provider的读取。

无市场分窗或经济结果，不晋升。历史PIT/独立前向资格仍false。下一步仅为完整实现独评与单独train-only无拟合GO；预检通过后正式训练/账户仍需最终评审和owner GO。代码测试通过不证明S/Z预测性或SOXX稳定超额。
